# LightGBM Training on EMBER Dataset
This notebook trains a LightGBM model on the EMBER dataset using `np.memmap` to handle the large data size without loading everything into RAM.
Training is done in chunks to ensure stability.

In [1]:
import os
import sys
import numpy as np
import lightgbm as lgb

# Add local venv site-packages to path so thrember can be imported
# In a notebook, __file__ is not defined, we use os.getcwd()
base_dir = os.getcwd()
if os.path.exists(os.path.join(base_dir, "venv", "Lib", "site-packages")):
    sys.path.append(os.path.join(base_dir, "venv", "Lib", "site-packages"))
elif os.path.exists(os.path.join(base_dir, ".venv", "Lib", "site-packages")):
    sys.path.append(os.path.join(base_dir, ".venv", "Lib", "site-packages"))

try:
    from thrember.features import PEFeatureExtractor
    print("Features extractor imported successfully.")
except ImportError:
    print("Warning: 'thrember' not found. Will use default feature dimension (2381).")
    class PEFeatureExtractor:
        dim = 2381

c:\Users\him\ember2024_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features extractor imported successfully.


In [ ]:
# Configuration
DATASET_DIR = r"Z:\ember2024_train_data"
CHUNK_SIZE = 10000

# LightGBM parameters optimized for EMBER
params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 1024,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "verbose": -1,
    "n_jobs": -1  # Use all CPU cores
}

# Indices of categorical features as defined by the EMBER dataset
categorical_features = [2, 3, 4, 5, 6, 701, 702]

In [3]:
print(f"Preparing to train from {DATASET_DIR}...")

X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

if not os.path.exists(X_path) or not os.path.exists(y_path):
    raise FileNotFoundError("Error: Training data not found!")

# Get the feature dimension
extractor = PEFeatureExtractor()
ndim = extractor.dim

# Calculate sizing
file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)

print(f"Total samples detected: {nrows}")
print(f"Feature dimensions: {ndim}")

# Open files in read-only memmap mode
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

# Split: 90% Training, 10% Validation
train_nrows = int(nrows * 0.9)
val_nrows = nrows - train_nrows
print(f"Training on {train_nrows} samples")
print(f"Validating on {val_nrows} samples (Skipped in this training loop)")

Preparing to train from Z:\ember2024_train_data...
Total samples detected: 5252000
Feature dimensions: 2568
Training on 4726800 samples
Validating on 525200 samples (Skipped in this training loop)


### Training Loop
The loop below iterates through the dataset in chunks. This is where the heavy lifting happens.

In [4]:
model = None
start_chunk_idx = 0

# RETRIEVE STATE LOGIC
ckpt_path = os.path.join(DATASET_DIR, "ember_model_checkpoint.txt")
state_path = os.path.join(DATASET_DIR, "ember_training_state.txt")

if os.path.exists(ckpt_path) and os.path.exists(state_path):
    print(f"Found checkpoint! Resuming training from {ckpt_path}...")
    try:
        model = lgb.Booster(model_file=ckpt_path)
        with open(state_path, "r") as f:
            start_chunk_idx = int(f.read().strip())
        print(f"Resuming from chunk index: {start_chunk_idx}")
    except Exception as e:
        print(f"Error loading checkpoint: {e}. Starting from scratch.")
        model = None
        start_chunk_idx = 0
else:
    print("No checkpoint found. Starting from scratch.")

print("\n" + "="*40)
print("🚀 TRAINING STARTED...")
print("="*40)
print(f"Starting training in chunks of {CHUNK_SIZE}...")

# Calculate total chunks for progress tracking
total_chunks = (train_nrows + CHUNK_SIZE - 1) // CHUNK_SIZE
current_chunk = 0

for start_idx in range(0, train_nrows, CHUNK_SIZE):
    # RESUME LOGIC: Skip chunks we've already processed
    if current_chunk < start_chunk_idx:
        current_chunk += 1
        continue

    end_idx = min(start_idx + CHUNK_SIZE, train_nrows)
    print(f"Training chunk {current_chunk}/{total_chunks}: {start_idx} to {end_idx}...")
    
    # Load ONLY this chunk into RAM
    X_chunk = np.array(X_memmap[start_idx:end_idx])
    y_chunk = np.array(y_memmap[start_idx:end_idx])
    
    # Filter out unlabeled data (-1)
    valid_idx = y_chunk != -1
    X_chunk = X_chunk[valid_idx]
    y_chunk = y_chunk[valid_idx]
    
    if len(y_chunk) == 0:
        print("  Skipping empty chunk")
        current_chunk += 1
        continue
        
    # Create LightGBM dataset for this chunk
    train_data = lgb.Dataset(
        X_chunk, 
        label=y_chunk, 
        categorical_feature=categorical_features,
        free_raw_data=False
    )
    
    # Train incrementally
    model = lgb.train(
        params,
        train_data,
        num_boost_round=50, 
        init_model=model,  
        keep_training_booster=True
    )
    
    current_chunk += 1
    
    # SAVE STATE LOGIC: Save checkpoint every 5 chunks
    if current_chunk % 5 == 0:
        model.save_model(ckpt_path)
        with open(state_path, "w") as f:
            f.write(str(current_chunk))
        print(f"  [Checkpoint saved to {ckpt_path}]")

print("\nTraining Loop Complete!")

Found checkpoint! Resuming training from Z:\ember2024_train_data\ember_model_checkpoint.txt...
Resuming from chunk index: 340

🚀 TRAINING STARTED...
Starting training in chunks of 10000...
Training chunk 340/473: 3400000 to 3410000...
Training chunk 341/473: 3410000 to 3420000...
Training chunk 342/473: 3420000 to 3430000...
Training chunk 343/473: 3430000 to 3440000...
Training chunk 344/473: 3440000 to 3450000...
  [Checkpoint saved to Z:\ember2024_train_data\ember_model_checkpoint.txt]
Training chunk 345/473: 3450000 to 3460000...
Training chunk 346/473: 3460000 to 3470000...
Training chunk 347/473: 3470000 to 3480000...
Training chunk 348/473: 3480000 to 3490000...
Training chunk 349/473: 3490000 to 3500000...
  [Checkpoint saved to Z:\ember2024_train_data\ember_model_checkpoint.txt]
Training chunk 350/473: 3500000 to 3510000...
Training chunk 351/473: 3510000 to 3520000...
Training chunk 352/473: 3520000 to 3530000...
Training chunk 353/473: 3530000 to 3540000...
Training chunk 35

In [5]:
# Save the final model
save_path = os.path.join(DATASET_DIR, "ember_model_full.txt")
print(f"Saving model to {save_path}...")
model.save_model(save_path)
print("Model saved successfully!")

Saving model to Z:\ember2024_train_data\ember_model_full.txt...
Model saved successfully!
